# Math Reasoning LLM - Operational Launcher

This notebook serves as the operational launcher for training, evaluation, and data generation.
It works seamlessly both **in Google Colab** (with T4/G4/A100 GPUs) and **in Antigravity IDE / local Jupyter**.

> **Architecture Rules:**
> 1. **No Training Logic in Notebook Cells:** All post-training, data synthesis, and evaluation runs inside modular scripts in `scripts/`.
> 2. **Persistence on Google Drive (Colab mode):** Artifacts (`checkpoints/`, `outputs/`, `logs/`, `data/`) are saved directly to `My Drive/math-reasoning-llm/`.
> 3. **High-Speed Model Caching:** Hugging Face model cache remains on fast local SSD disk (`/content/hf_cache` on Colab, or local `.cache/`), NEVER on Google Drive.
> 4. **Zero-Secret Commits:** Hugging Face token is loaded directly from environment or Secrets (`HF_TOKEN`).

## Step 1: Environment Detection & Persistence Setup
Detects whether running in Google Colab (mounts Google Drive) or local workstation, and sets the root directory.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure current working directory is the repository root, not notebooks/
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
print(f"Active working directory: {Path.cwd()}")

# Check if running inside Google Colab
IS_COLAB = "google.colab" in sys.modules
try:
    from google.colab import drive
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    print("Running in Google Colab. Mounting Google Drive...")
    drive.mount('/content/drive')
    DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/math-reasoning-llm')
    for subfolder in ['checkpoints', 'outputs', 'logs', 'data']:
        (DRIVE_PROJECT_ROOT / subfolder).mkdir(parents=True, exist_ok=True)
    print(f"Google Drive mounted successfully at: {DRIVE_PROJECT_ROOT}")
else:
    print("Running in local/IDE environment. Using local workspace storage.")
    for subfolder in ['checkpoints', 'outputs', 'logs', 'data']:
        Path(subfolder).mkdir(parents=True, exist_ok=True)
    print("Local directories verified.")

## Step 2: Repository Branch Synchronization
Ensures latest project code is checked out.

In [ ]:
REPO_URL = "https://github.com/NashLucas/math-reasoning-llm.git"
REPO_DIR = "/content/math-reasoning-llm"
BRANCH = "phase-2-generation-and-baseline"

if IS_COLAB:
    if not os.path.exists(REPO_DIR):
        print(f"Cloning {REPO_URL}...")
        !git clone {REPO_URL} {REPO_DIR}
        %cd {REPO_DIR}
        !git checkout {BRANCH}
    else:
        print(f"Updating repository in {REPO_DIR}...")
        %cd {REPO_DIR}
        !git fetch origin
        !git checkout {BRANCH}
        !git pull origin {BRANCH}
    %cd {REPO_DIR}
else:
    print("Local workspace already loaded.")

!git status

## Step 3: Hardware Diagnostics & Environment Configuration
Inspects GPU hardware (CUDA, VRAM) and configures HF cache.

In [ ]:
try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

# 1. HF Token
if IS_COLAB:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        os.environ['HF_TOKEN'] = hf_token
        os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
        print("HF token loaded from Colab Secrets.")
    except Exception:
        print("NOTE: HF_TOKEN not set in Colab Secrets (optional for public models like Qwen / DeepSeek-Math).")
    os.environ['HF_HOME'] = '/content/hf_cache'
    os.environ['APP_PROFILE'] = 'colab'
    os.makedirs('/content/hf_cache', exist_ok=True)

# 2. Diagnostics
print("=" * 60)
print("HARDWARE DIAGNOSTICS")
print("=" * 60)
if HAS_TORCH and torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Detected: {gpu_name}")
    print(f"VRAM: {vram_gb:.2f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"PyTorch Version: {torch.__version__}")
else:
    print("CPU Runtime detected (No CUDA GPU active).")
    if IS_COLAB:
        print("To enable GPU on Colab: Runtime -> Change runtime type -> T4 GPU.")
print("=" * 60)

## Step 4: Install Dependencies

In [ ]:
if IS_COLAB:
    !pip install -r requirements.txt
else:
    print("Dependencies managed by local environment.")

## Step 5: Verify Configurations & Stage Paths

In [ ]:
!python src/config.py

## Step 5b: Prepare GSM8K Splits on the Profile Data Directory

Generated data (`raw/`, `processed/`, `safety/`) lives in the profile data directory: `data/` locally, `My Drive/math-reasoning-llm/data/` on Colab, so it survives a disconnect.
Data preparation is deterministic (bit-identical splits), so this step is skipped when the processed splits already exist on Drive.
Evaluation never falls back to the 20-problem sample set on a real run: a missing split stops the run instead.

In [ ]:
from src.config import ConfigManager

_processed_val = ConfigManager().data_path('processed', 'val', 'val.parquet')
if _processed_val.exists():
    print(f'Processed splits found: {_processed_val.parent.parent}')
else:
    print(f'Processed splits missing at {_processed_val.parent.parent}; preparing data...')
    !python scripts/prepare_data.py


## Step 5c: Day 5 — vLLM Check (run once per runtime type, e.g. on G4)

Installs `requirements-vllm.txt` and runs a tiny generation with the dev model.
Writes the verdict to `outputs/env_checks/vllm_check.json` (Drive under colab); log it in `EXPERIMENTS.md`.
If the verdict is `use_hf_fallback`, apply the vLLM row of the plan's fallback rules.

In [ ]:
if HAS_TORCH and torch.cuda.is_available():
    !pip install -q -r requirements-vllm.txt -c requirements.txt
    !python scripts/check_vllm.py
else:
    print('vLLM needs a CUDA GPU; skipping (CPU / local).')


## Step 5d: Phase 0 Gate — Training Resumes After a Disconnect

Trains a tiny GPT-2 with the Hugging Face Trainer for 50 steps (checkpoint every 10) into `checkpoints/resume_test` (Drive under colab). A CPU runtime is enough.

**Real disconnect test (the gate):**
1. Run cell **1)**. It starts a fresh test and prints `step N/50` and `saved checkpoint-N` as it goes.
2. After `saved checkpoint-10` and **before step 50**, use **Runtime → Disconnect and delete runtime** (not "Restart session": the machine and its disk must be gone).
3. **Connect** to a new runtime, re-run Steps 1–4, then run cell **2)**. It must print `Resuming from checkpoint-N` with N > 0 and finish at step 50.
4. Run cell **3)**. It must print `PHASE 0 RESUME GATE: PASSED`, including `[ok] Resumed on a different machine`.

Never run cell 1) after the disconnect: `--fresh` deletes the checkpoints. Re-running cell 2) after training finished does nothing.
Cell **4)** is an optional quick check: a hard process kill and resume on the same machine (`[info]` for the machine check).

In [ ]:
# 1) FIRST RUN ONLY: start a fresh test, then disconnect and delete the runtime mid-run
!python scripts/test_resume_training.py --mode run --fresh --step_delay 3.0


In [ ]:
# 2) AFTER RECONNECTING to a new runtime: resume from the last checkpoint on Drive
!python scripts/test_resume_training.py --mode run --step_delay 0.3


In [ ]:
# 3) Verify the gate
!python scripts/test_resume_training.py --mode verify


In [ ]:
# 4) Optional quick check: hard process kill + resume on the same machine (starts fresh)
!python scripts/test_resume_training.py --mode auto


## Step 6: Day 11 Verification — Development Model Generation & Chunk Resumption

- On GPU (Colab T4): runs with `--use_real_model` to load `Qwen/Qwen2.5-1.5B-Instruct` on GPU.
- On CPU (Local): runs without `--use_real_model` to test stop rules and resumption with the mock generator.

In [ ]:
# If on GPU runtime, use real model; otherwise use mock generator
if HAS_TORCH and torch.cuda.is_available():
    print("Running Day 11 verification with real Qwen2.5-1.5B-Instruct on GPU...")
    !python scripts/test_generation_engine.py --use_real_model --model_name "Qwen/Qwen2.5-1.5B-Instruct"
else:
    print("Running Day 11 verification in mock mode (CPU / local test)...")
    !python scripts/test_generation_engine.py

## Step 7: Day 12 Verification — Evaluation Engine & Metrics on Sample Data

Tests `scripts/evaluate.py` end-to-end:
- Runs greedy evaluation on the 20 sample problems (`val_sample.parquet`)
- Computes math metrics (accuracy, format compliance, fallback rate, token averages)
- Generates `eval_results.json` and `run_info.json` in `outputs/baseline/`

In [ ]:
if HAS_TORCH and torch.cuda.is_available():
    print('Running Day 12 evaluation on sample data with real Qwen2.5-1.5B on GPU...')
    !python scripts/evaluate.py --stage baseline --preset greedy --split sample --limit 20
else:
    print('Running Day 12 evaluation on sample data in mock mode (CPU / local test)...')
    !python scripts/evaluate.py --stage baseline --preset greedy --split sample --limit 20 --use_mock


## Step 8: Day 13 — Development Model Baseline (500 Validation Problems)

Runs greedy and self-consistency decoding across all 500 validation problems (`val.parquet`):
- Writes to `outputs/baseline/Qwen2.5-1.5B-Instruct/` (one folder per model):
  `eval_results_val_{greedy,self_consistency}.json`, `generations_*.jsonl` (+ `.meta.json`), `run_info.json`
- High-throughput inference via vLLM (or Hugging Face batched generation)

In [ ]:
if HAS_TORCH and torch.cuda.is_available():
    print('Running Day 13 development baseline on 500 validation problems (GPU)...')
    !python scripts/run_dev_baseline.py --split val
else:
    print('Running Day 13 development baseline in mock mode (CPU / local test)...')
    !python scripts/run_dev_baseline.py --split val --limit 20 --use_mock


## Step 9: Day 13 — Output Audit & Parser Quality Review

Inspects 20 model reasoning completions to verify answer extraction, format compliance, and equivalence checking.
Generates `reports/day13_output_review.md`.

In [ ]:
!python scripts/review_outputs.py --count 20


## Step 10: Day 14 — 7B Baselines (3 Models x 2 Presets on 500 Validation Problems)

Executes baseline evaluations for all three DeepSeek-Math 7B models on `val.parquet` (500 problems):
1. `deepseek-ai/deepseek-math-7b-base` (8-shot few-shot prompt)
2. `deepseek-ai/deepseek-math-7b-instruct` (chat template)
3. `deepseek-ai/deepseek-math-7b-rl` (chat template)

Under `greedy` and `self_consistency` decoding presets.
Outputs 6 result sets in `outputs/baseline/<model_slug>/` and summary table `outputs/baseline/baseline_comparison.md`.

**First G4 run checklist** (vLLM memory release cannot be tested without a GPU):
1. Step 5c ran on this runtime, so vLLM is installed; the log says `Auto-selected generation backend: 'vllm'`.
2. The log shows `prompt_format=few_shot` for the base model and `prompt_format=chat` for instruct and rl.
3. After each run the log prints `GPU memory after release: X GiB free of Y GiB`; X should be close to Y.
   If run 2 fails with CUDA out of memory, run one model per process instead, e.g.
   `!python scripts/run_7b_baselines.py --split val --models deepseek-ai/deepseek-math-7b-base` (then instruct, then rl).
4. In `baseline_comparison.md`, `Truncated (%)` should be near 0; a high value means `max_new_tokens` is too low.
5. A disconnect is safe: rerun the same cell and finished chunks are skipped. Changing the model, prompt or
   decoding settings for an existing folder raises an error instead of mixing runs.

In [ ]:
if HAS_TORCH and torch.cuda.is_available():
    print('Running Day 14 7B baselines on GPU (G4/T4)...')
    !python scripts/run_7b_baselines.py --split val
else:
    print('Running Day 14 7B baselines in mock mode (CPU / local test)...')
    !python scripts/run_7b_baselines.py --split val --limit 20 --use_mock


## Step 11: Day 15 — Safety Prompt Set & Llama Guard Screening

1. Builds `data/safety/` (50 JailbreakBench harmful prompts, 5 per category, + 50 benign GSM8K train questions, seed 42).
2. Runs the safety unit tests.
3. On a GPU with at least 20 GB VRAM (L4 / G4 / A100), generates development-model responses and screens them with `meta-llama/Llama-Guard-3-8B`.
   Llama Guard 3 is gated: `HF_TOKEN` must belong to an account with approved access. A T4 (15 GB) cannot hold it in bf16.

Outputs `outputs/baseline/safety/<model_slug>/safety_results.json`, `safety_assessments.jsonl` and `run_info.json`.

In [ ]:
!python scripts/prepare_safety_data.py
!python -m unittest tests/test_safety.py

if HAS_TORCH and torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    if vram_gb >= 20:
        print(f'Running Day 15 safety evaluation on the development model ({vram_gb:.0f} GB GPU)...')
        !python scripts/run_safety_eval.py
    else:
        print(f'GPU has {vram_gb:.0f} GB; Llama Guard 3 8B needs ~16 GB plus headroom. Switch to L4/G4/A100.')
else:
    print('Running Day 15 safety evaluation in mock mode (CPU / local test)...')
    !python scripts/run_safety_eval.py --use_mock --limit 10 --output_dir outputs/mock/safety


## Step 12: Day 15 — Manual Review of 30 Outputs per 7B Model

Writes one review report per model to `reports/day15_review_<model_slug>.md` from the **real** Step 10 greedy generations.
Read every row and check that the parsed answer matches what you would read; add every parser miss to `tests/test_answer_extraction.py` before fixing it.

In [ ]:
from src.config import ConfigManager

_baseline_dir = ConfigManager().get_stage_paths('baseline')['outputs']
for slug in ['deepseek-math-7b-base', 'deepseek-math-7b-instruct', 'deepseek-math-7b-rl']:
    gen_file = _baseline_dir / slug / 'generations_val_greedy.jsonl'
    !python scripts/review_outputs.py --input_file "{gen_file}" --count 30 --output_report reports/day15_review_{slug}.md


## Step 13: Phase 3 Day 16 — Reasoning Prompts Verification (CoT, Solve-then-Verify, Alternative-Method)

Tests all three reasoning prompt strategies on 10 sample problems with the development model:
1. `chain_of_thought`: direct step-by-step mathematical reasoning.
2. `solve_then_verify`: two-stage generation (initial solution followed by verification/sanity check).
3. `alternative_method`: alternative formulation (algebraic modeling, working backwards, or proportional reasoning).

Verifies 100% format compliance with the canonical answer contract (`#### <number>`).
Writes results to `reports/day16_prompt_verification.json` and `reports/day16_reasoning_prompts.md`.

In [ ]:
if HAS_TORCH and torch.cuda.is_available():
    print('Testing Day 16 reasoning prompts on sample problems with real model on GPU...')
    !python scripts/test_reasoning_prompts.py --use_real_model --num_problems 10
else:
    print('Testing Day 16 reasoning prompts in mock mode (CPU / local)...')
    !python scripts/test_reasoning_prompts.py --num_problems 10


## Step 14: Phase 3 Day 17 — Synthetic Reasoning Trace Sampling (100-Problem Dev Run)

Samples 4 reasoning traces per problem across 100 problems from the 3,000-problem training subset.
Records trace ID, rendered prompt, completion, parsed answer, extraction tier, correctness flag, token counts, and UTC timestamp.
Writes chunked, resumable JSONL to `data/synthetic/generated/` and yield report to `reports/day17_synthetic_generation.md`.

In [ ]:
if HAS_TORCH and torch.cuda.is_available():
    print('Running Day 17 trace sampling with real model on GPU...')
    !python scripts/generate_reasoning_data.py --limit 100 --samples_per_problem 4 --use_real_model
else:
    print('Running Day 17 trace sampling in mock mode (CPU / local)...')
    !python scripts/generate_reasoning_data.py --limit 100 --samples_per_problem 4 --use_mock


## Step 15: Phase 3 Day 18 — 7B Instruct Chain-of-Thought Sampling (3,000 Problems x 4 Samples = 12,000 Traces on G4)

Generates 4 chain-of-thought samples for each of the 3,000 problems from the training subset using `deepseek-ai/deepseek-math-7b-instruct` under the `sampled` preset (temperature 0.7, top-p 0.95, max_new_tokens 512).
Applies the correctness filter (`src/data/curation.py`) to verify answer equivalence and remove truncated or degenerate traces.
Saves chunked JSONL to `data/synthetic/generated/` and logs yield metrics.

In [ ]:
if HAS_TORCH and torch.cuda.is_available():
    print('Running Day 18 7B Instruct Chain-of-Thought sampling on GPU (G4)...')
    !python scripts/generate_reasoning_data.py --model_name "deepseek-ai/deepseek-math-7b-instruct" --samples_per_problem 4 --prompt_types "chain_of_thought" --use_real_model --report_name "day18_cot_sampling"
else:
    print('Running Day 18 CoT sampling in mock mode (CPU / local test)...')
    !python scripts/generate_reasoning_data.py --limit 50 --samples_per_problem 4 --prompt_types "chain_of_thought" --use_mock --report_name "day18_cot_sampling"


## Step 16: Phase 3 Day 19 — Reasoning Trace Yield Analysis & Per-Problem Pass Rates

Computes and saves:
1. Per-problem pass rates (correct samples out of 4) saved to `data/synthetic/generated/problem_pass_rates.json` and `.parquet` (reused in Phase 6 for GRPO difficulty filtering).
2. Overall trace yield, problem solve rate, unsolved count, and formatting failures.
3. Diagnostic markdown report in `reports/day19_yield_analysis.md` and JSON metrics in `reports/day19_yield_analysis_report.json`.
4. Curated SFT dataset of verified correct traces in `data/synthetic/curated/traces_curated_sft.jsonl`.

In [ ]:
if HAS_TORCH and torch.cuda.is_available():
    print('Running Day 19 Yield Analysis on generated 7B CoT traces...')
    !python scripts/analyze_reasoning_yield.py --traces_file data/synthetic/generated/traces_deepseek-math-7b-instruct_chain_of_thought.jsonl --output_dir data/synthetic/generated/ --save_curated
else:
    print('Running Day 19 Yield Analysis in mock/dev mode...')
    !python scripts/analyze_reasoning_yield.py --output_dir data/synthetic/generated/ --save_curated --mock


## Step 17: Phase 3 Day 20 — Verification & Alternative-Method Trace Generation (on Solved CoT Problems)

For every problem with at least one correct chain-of-thought trace (from Day 19 pass rates):
1. Generates 1 `solve_then_verify` trace.
2. Generates 1 `alternative_method` trace.
3. Filters both using `CorrectnessFilter` (`src/data/curation.py`) and slices per-prompt JSONLs for Day 21 judge scoring.

In [ ]:
if HAS_TORCH and torch.cuda.is_available():
    print('Running Day 20 Verification and Alternative-Method trace sampling on GPU (G4)...')
    !python scripts/generate_reasoning_data.py \
        --model_name "deepseek-ai/deepseek-math-7b-instruct" \
        --prompt_types "solve_then_verify,alternative_method" \
        --samples_per_problem 1 \
        --only_solved \
        --use_real_model \
        --report_name "day20_multi_strategy_traces"
else:
    print('Running Day 20 trace sampling in mock/dev mode...')
    !python scripts/generate_reasoning_data.py \
        --limit 50 \
        --prompt_types "solve_then_verify,alternative_method" \
        --samples_per_problem 1 \
        --only_solved \
        --use_mock \
        --report_name "day20_multi_strategy_traces"


## Step 18: Phase 3 Day 21 — LLM Judge Rubric Evaluation & Benchmark Calibration

Evaluates reasoning traces against the 3-dimension rubric (Completeness, Verification Quality, and Novelty).
Runs calibration on the 20 ground-truth benchmark traces in `data/calibration/manual_scores_20.json` to verify Pearson correlation $r \ge 0.60$.

In [ ]:
if HAS_TORCH and torch.cuda.is_available():
    print('Running Day 21 Judge Calibration with model on GPU (G4)...')
    !python scripts/score_traces_judge.py \
        --model_name "deepseek-ai/deepseek-math-7b-instruct" \
        --calibrate \
        --use_real_model \
        --report_name "day21_judge_calibration"
else:
    print('Running Day 21 Judge Calibration in mock/dev mode...')
    !python scripts/score_traces_judge.py \
        --calibrate \
        --use_mock \
        --report_name "day21_judge_calibration"


## Step 19: Phase 3 Day 22 — Full-Scale LLM Judge Scoring & Distribution Analysis (7B Instruct on G4)

Scores all correct reasoning traces across all three prompt strategies (`chain_of_thought`, `solve_then_verify`, `alternative_method`) using `deepseek-ai/deepseek-math-7b-instruct` on GPU (G4).
Strictly enforces Design Rule 4 (correctness filters first, judge ranks second).
Computes calibration metrics on the 20 ground-truth benchmark traces ($r \ge 0.60$), score spread statistics, and assigns within-problem relative ranks (`rank_within_problem`) for Day 23 curation.

In [ ]:
if HAS_TORCH and torch.cuda.is_available():
    print('Running Day 22 Judge Scoring on G4 with deepseek-ai/deepseek-math-7b-instruct...')
    !python scripts/score_traces_judge.py \
        --model_name "deepseek-ai/deepseek-math-7b-instruct" \
        --traces_dir "data/synthetic/generated" \
        --output_file "data/synthetic/generated/traces_scored.jsonl" \
        --split_by_prompt_type \
        --use_real_model \
        --check_calibration \
        --report_name "day22_judge_scoring"
else:
    print('Running Day 22 Judge Scoring in mock/dev mode...')
    !python scripts/score_traces_judge.py \
        --traces_dir "data/synthetic/generated" \
        --output_file "data/synthetic/generated/traces_scored.jsonl" \
        --split_by_prompt_type \
        --use_mock \
        --check_calibration \
        --report_name "day22_judge_scoring"


## Step 20: Phase 3 Day 23 — SFT Reasoning Dataset Curation & Contamination Gate

Executes final SFT reasoning dataset curation from scored synthetic traces and human gold solutions:
- Retains at most 2 traces per problem, prioritizing prompt strategy diversity (e.g. 1 alternative method + 1 CoT) and top judge score.
- Filters near-duplicates (Jaccard $\ge 0.85$), overlength traces (>512 tokens), and degenerate text.
- Uses cleaned human gold solutions for unsolved problems to ensure 100% problem coverage.
- Asserts strict zero contamination against validation and test splits.
- Generates `data/synthetic/curated/traces_curated_sft.jsonl` and diagnostic reports.

In [ ]:
print('Running Day 23 Reasoning Dataset Curation...')
!python scripts/curate_data.py \
    --traces_file "data/synthetic/generated/traces_scored.jsonl" \
    --output_file "data/synthetic/curated/traces_curated_sft.jsonl" \
    --max_traces_per_problem 2 \
    --report_name "day23_curation"


## Step 21: Phase 3 Day 24 — Reasoning Trace Inspection & Systematic Audit

Executes qualitative inspection and audit across curated and rejected reasoning traces:
- Generates 100-trace qualitative audit (50 representative kept + 50 representative rejected traces).
- Analyzes response length distribution percentiles (P25, P50, P75, P95) across all prompt strategies.
- Measures judge score spread and validates within-problem relative ranking.
- Generates `reports/day24_inspection_report.md` and `notebooks/02_inspect_reasoning_traces.ipynb` analysis.

In [ ]:
print('Running Day 24 Trace Inspection & Systematic Audit...')
!python scripts/inspect_reasoning_traces.py \
    --curated_file "data/synthetic/curated/traces_curated_sft.jsonl" \
    --scored_file "data/synthetic/generated/traces_scored.jsonl" \
    --pass_rates_file "data/synthetic/generated/problem_pass_rates.json" \
    --report_name "day24_inspection_report"


## Step 22: Phase 3 Day 25 — Dataset Freeze, Cryptographic Hashing & Gate v0.4 Verification

Executes the final Phase 3 freeze protocol:
- Enforces canonical format hygiene (`\n#### <number>`) and deterministic sorting.
- Asserts strict zero contamination against validation (`500`) and test (`1,319`) problems.
- Computes SHA-256 cryptographic checksum and creates `traces_curated_sft_manifest.json`.
- Evaluates Phase 3 Gate criteria and generates `reports/day25_freeze_report.md` for `v0.4` release.

In [ ]:
print('Running Day 25 SFT Dataset Freeze & Gate v0.4 Verification...')
!python scripts/freeze_curated_data.py \
    --curated_file "data/synthetic/curated/traces_curated_sft.jsonl" \
    --manifest_file "data/synthetic/curated/traces_curated_sft_manifest.json" \
    --splits_dir "data/splits" \
    --reports_dir "reports" \
    --report_name "day25_freeze_report"


## Step 23: Phase 4 Day 26 — SFT Tokenization Hand-Check & Model Training

Executes Phase 4 Day 26 SFT workflow:
- Runs the Day 26 hand check on 3 curated examples to verify completion-only loss masking and unmasked EOS supervision.
- Loads `configs/lora_config.yaml` (r=16, alpha=32, all linear layers) and `configs/training_args.json` (bf16, effective batch 32, cosine schedule).
- Starts SFT LoRA training on `data/synthetic/curated/traces_curated_sft.jsonl` with automatic checkpoint resumption.

In [ ]:
print('=== Day 26: SFT Tokenization Hand Check ===')
!python scripts/verify_sft_tokenization.py \
    --curated_file "data/synthetic/curated/traces_curated_sft.jsonl" \
    --model_name "Qwen/Qwen2.5-1.5B-Instruct" \
    --num_examples 3 \
    --reports_dir "reports" \
    --report_name "day26_sft_tokenization_check"

print('\n=== Day 26: Launching SFT LoRA Training ===')
!python scripts/train_sft.py \
    --model_name_or_path "deepseek-ai/deepseek-math-7b-base" \
    --curated_file "data/synthetic/curated/traces_curated_sft.jsonl" \
    --output_dir "checkpoints/sft" \
    --adapter_dir "outputs/sft/adapter_model" \
    --lora_config "configs/lora_config.yaml" \
    --training_args "configs/training_args.json"


## Step 24: Phase 4 Day 27 — SFT Overfitting Sanity Test & Periodic Validation Callback

Executes Phase 4 Day 27 SFT sanity and evaluation workflow:
- Runs the 20-example overfitting sanity test (`scripts/test_sft_overfit.py`) verifying loss decay and format compliance.
- Validates that the SFT training prompt matches the zero-shot inference prompt exactly.
- Launches SFT training with `ValidationEvalCallback` logging greedy accuracy and format compliance every 100 steps.

In [ ]:
print('=== Day 27: SFT Overfitting Sanity Test (20 Examples) ===')
!python scripts/test_sft_overfit.py \
    --model_name_or_path "Qwen/Qwen2.5-1.5B-Instruct" \
    --curated_file "data/synthetic/curated/traces_curated_sft.jsonl" \
    --num_examples 20 \
    --max_steps 30 \
    --output_dir "checkpoints/sft_overfit" \
    --reports_dir "reports"

print('\n=== Day 27: Launching SFT Training with Periodic Validation Callback ===')
!python scripts/train_sft.py \
    --model_name_or_path "deepseek-ai/deepseek-math-7b-base" \
    --curated_file "data/synthetic/curated/traces_curated_sft.jsonl" \
    --output_dir "checkpoints/sft" \
    --adapter_dir "outputs/sft/adapter_model" \
    --lora_config "configs/lora_config.yaml" \
    --training_args "configs/training_args.json" \
    --eval_steps 100 \
    --eval_limit 100 \
    --val_file "data/splits/val.parquet" \
    --generations_dir "logs/generations"


## Step 25: Phase 4 Day 28 — Learning-Rate Sweep on the Development Model

Executes Phase 4 Day 28 learning-rate sweep:
- Runs three short trainings of the development model (`Qwen/Qwen2.5-1.5B-Instruct`) at learning rates `5e-5`, `1e-4`, and `2e-4`.
- Uses `ValidationEvalCallback` to evaluate greedy accuracy and format compliance across all three runs.
- Generates comparative training curves and audit reports in `reports/`.
- Automatically preserves the winning development adapter at `outputs/sft/adapter_model_dev/` for Phase 6 trial runs.

In [ ]:
print('=== Day 28: Learning-Rate Sweep on Development Model ===')
!python scripts/sweep_learning_rates.py \
    --model_name_or_path "Qwen/Qwen2.5-1.5B-Instruct" \
    --curated_file "data/synthetic/curated/traces_curated_sft.jsonl" \
    --val_file "data/splits/val.parquet" \
    --learning_rates 5e-5 1e-4 2e-4 \
    --max_steps 50 \
    --eval_steps 25 \
    --eval_limit 100 \
    --output_base_dir "outputs/sft/lr_sweep" \
    --reports_dir "reports" \
    --best_adapter_dir "outputs/sft/adapter_model_dev"


## Step 26: Phase 4 Day 29 — 7B SFT LoRA Training on G4 with Resumption

Executes Phase 4 Day 29 production SFT training of `deepseek-ai/deepseek-math-7b-base`:
- Uses the winning learning rate `1e-4` from the Day 28 sweep, effective batch size 32, and LoRA $r=16, \alpha=32$.
- Automatically detects and resumes from the latest completed checkpoint on Google Drive (`checkpoints/sft`) if a Colab session disconnects.
- Integrates `ValidationEvalCallback` to evaluate greedy accuracy and format compliance on 100 validation problems every 50 steps.
- Persists all step checkpoints and the final trained adapter to Google Drive (`outputs/sft/adapter_model`).
- Generates `reports/day29_7b_sft_report.md` and training curves.

In [ ]:
print('=== Day 29: 7B SFT LoRA Training with Automatic Resumption ===')
if HAS_TORCH and torch.cuda.is_available():
    print('Launching 7B SFT training on GPU (G4/A100)...')
    !python scripts/train_7b_sft.py \
        --model_name_or_path "deepseek-ai/deepseek-math-7b-base" \
        --curated_file "data/synthetic/curated/traces_curated_sft.jsonl" \
        --learning_rate 1e-4 \
        --eval_steps 50 \
        --eval_limit 100
else:
    print('Running Day 29 7B SFT training in dry-run mode (CPU / local)...')
    !python scripts/train_7b_sft.py \
        --sample_mode \
        --max_steps 5 \
        --eval_steps 2 \
        --eval_limit 10


## Step 27: Phase 4 Day 30 — SFT Checkpoint Evaluation, Selection & Phase 4 Gate (v0.5)

Executes Phase 4 Day 30 checkpoint selection and gate evaluation:
- Evaluates every saved checkpoint on all 500 validation problems (`val.parquet`) with greedy decoding.
- Ranks checkpoints by validation greedy accuracy (Design Rule 3), copies the winning adapter to `outputs/sft/adapter_model/`, and records provenance.
- Evaluates the best adapter model with self-consistency (maj@8).
- Verifies Phase 4 Gate: greedy validation accuracy beats 8-shot 7B base model, format compliance >= 98%, low truncation.
- Generates `reports/sft_training.md`, `reports/day30_best_checkpoint.json`, and comparison curves.
- Tags release `v0.5` upon passing the gate.

In [ ]:
print('=== Day 30: Checkpoint Evaluation, Selection & Phase 4 Gate Verification ===')
if HAS_TORCH and torch.cuda.is_available():
    print('Evaluating checkpoints on GPU (G4/A100)...')
    !python scripts/pick_best_checkpoint.py \
        --model_name_or_path "deepseek-ai/deepseek-math-7b-base" \
        --val_split "val"
else:
    print('Running Day 30 checkpoint evaluation in mock mode (CPU / local)...')
    !python scripts/pick_best_checkpoint.py \
        --use_mock \
        --limit 10


## Step 28: Phase 5 Day 31 — Error Analysis & Taxonomy Classification

Executes Phase 5 Day 31 mathematical error analysis:
- Evaluates mathematical reasoning errors against a rigorous 7-category taxonomy:
  1. `parse_error`: empty output, missing final numeric answer, unparseable (rule-based).
  2. `truncation`: output cut off by token length limits or abrupt incomplete sentences (rule-based).
  3. `arithmetic_slip`: correct setup/equations, calculation or elementary computation mistake.
  4. `misread_question`: misread numbers, entities, or solved for wrong quantity.
  5. `wrong_setup`: incorrect formula, conceptual approach, or problem formulation.
  6. `incomplete_reasoning`: stopped prematurely, skipped steps, or guessed without derivation.
  7. `other`: degenerate repetition, language mixing, or incoherent failure.
- Generates `reports/error_analysis_sft.json` and Markdown report `reports/error_analysis_sft.md`.

In [ ]:
print('=== Day 31: Error Analysis & Taxonomy Classification ===')
if HAS_TORCH and torch.cuda.is_available():
    print('Classifying reasoning errors with 7B instruct teacher on GPU (G4/A100)...')
    !python scripts/error_analysis.py \
        --input_file 'outputs/sft/adapter_model/eval_val_greedy/eval_results.json' \
        --model_name_or_path 'deepseek-ai/deepseek-math-7b-instruct' \
        --output_file 'reports/error_analysis_sft.json' \
        --report_file 'reports/error_analysis_sft.md'
else:
    print('Running Day 31 error classification in sample/mock mode (CPU / local)...')
    !python scripts/error_analysis.py \
        --sample_mode \
        --output_file 'reports/error_analysis_sft.json' \
        --report_file 'reports/error_analysis_sft.md'


## Step 29: Phase 5 Day 32 — Classify SFT Model Errors on Analysis Half & 30-Example Calibration

Executes Phase 5 Day 32 error classification & human agreement calibration:
- Evaluates SFT adapter model with greedy decoding on `data/processed/val/val_analysis.parquet` (250 problems).
- Strictly enforces Design Rule 2 (The Test-Set Wall): test set is NEVER touched or accessed.
- Automatically classifies reasoning errors across the 7-category taxonomy (`arithmetic_slip`, `misread_question`, `wrong_setup`, `incomplete_reasoning`, `parse_error`, `truncation`, `other`).
- Runs 30-sample human calibration benchmark (`data/sample/labeled_30_errors.jsonl`) verifying Day 32 Gate (>= 80% agreement).
- Produces `reports/error_label_agreement_30.json`, `reports/error_label_agreement_30.md`, and updates `reports/error_analysis_sft.json` & `reports/error_analysis_sft.md`.

In [ ]:
print('=== Day 32: Classify SFT Model Errors on Validation Analysis & Calibration ===')
if HAS_TORCH and torch.cuda.is_available():
    print('Running Day 32 full pipeline on GPU (G4/A100)...')
    !python scripts/run_day32_error_analysis.py \
        --val_analysis_file 'data/processed/val/val_analysis.parquet' \
        --calibration_file 'data/sample/labeled_30_errors.jsonl' \
        --adapter_dir 'outputs/sft/adapter_model' \
        --output_dir 'outputs/val_analysis_eval' \
        --reports_dir 'reports'
else:
    print('Running Day 32 error analysis in mock/sample mode (CPU / local)...')
    !python scripts/run_day32_error_analysis.py \
        --sample_mode \
        --val_analysis_file 'data/processed/val/val_analysis.parquet' \
        --calibration_file 'data/sample/labeled_30_errors.jsonl' \
        --reports_dir 'reports'

## Step 30: Phase 5 Day 33 — Error Question Semantic Clustering & Topic Reporting

Executes Phase 5 Day 33 error clustering and report generation:
- Embeds reasoning error questions using `sentence-transformers/all-MiniLM-L6-v2` (or TF-IDF fallback on CPU).
- Clusters errors into distinct mathematical semantic topics (percentages, rates, money, time, counting).
- For each cluster, extracts top discriminating keywords and assigns domain topic labels.
- Selects 2 representative examples closest to each cluster centroid for qualitative failure inspection.
- Emits `reports/error_clusters.json` and updates `reports/error_analysis_sft.md` with full cluster profiles and representative examples.

In [ ]:
print('=== Day 33: Error Question Semantic Clustering & Topic Reporting ===')
if HAS_TORCH and torch.cuda.is_available():
    print('Running Day 33 clustering with sentence-transformers on GPU (G4/A100)...')
    !python scripts/run_day33_clustering.py \
        --input_file 'reports/error_analysis_sft.json' \
        --output_clusters_file 'reports/error_clusters.json' \
        --report_file 'reports/error_analysis_sft.md' \
        --num_clusters 5 \
        --embedding_model 'sentence-transformers/all-MiniLM-L6-v2'
else:
    print('Running Day 33 clustering in sample/mock mode (CPU / local)...')
    !python scripts/run_day33_clustering.py \
        --sample_mode \
        --input_file 'data/sample/labeled_30_errors.jsonl' \
        --output_clusters_file 'reports/error_clusters.json' \
        --report_file 'reports/error_analysis_sft.md' \
        --num_clusters 5

## Step 31: Phase 5 Day 34 — Targeted Example Selection for SFT Round 2

Executes Phase 5 Day 34 targeted training example selection for error mitigation:
- Loads candidate training pool (`data/processed/train/train.parquet`, 6,973 problems).
- Enforces Design Rule 2 (The Test-Set Wall) against 1,819 forbidden validation/test IDs.
- Computes semantic embeddings for training pool questions and error category centroids.
- Selects 20–50 training problems most similar to each error category via cosine similarity.
- Attaches curated reasoning trace from `data/synthetic/curated/traces_curated_sft.jsonl` (or canonical `cleaned_solution`).
- Exports targeted dataset to `data/targeted/targeted_examples.jsonl`, `targeted_examples.parquet`, and `manifest.json`.

In [ ]:
print('=== Day 34: Targeted Example Selection for SFT Round 2 ===')
if HAS_TORCH and torch.cuda.is_available():
    print('Running Day 34 targeted selection with sentence-transformers on GPU (G4/A100)...')
    !python scripts/run_day34_selection.py \
        --error_file 'reports/error_analysis_sft.json' \
        --train_pool 'data/processed/train/train.parquet' \
        --curated_traces 'data/synthetic/curated/traces_curated_sft.jsonl' \
        --output_dir 'data/targeted' \
        --target_per_category 30 \
        --embedding_model 'sentence-transformers/all-MiniLM-L6-v2'
else:
    print('Running Day 34 targeted selection in sample/mock mode (CPU / local)...')
    !python scripts/run_day34_selection.py \
        --sample_mode \
        --output_dir 'data/targeted' \
        --target_per_category 5


## Step 32: Phase 5 Day 35 — Prepare & Freeze Round-2 SFT Dataset

Combines Phase 3 curated reasoning traces with Phase 5 Day 34 error-targeted training examples:
- Supports `--duplicate_targeted` (default: True) for 2x representation of error mitigation problems.
- Enforces Design Rule 1 (canonical answer contract ending in `#### <number>`).
- Strictly asserts Design Rule 2 (The Test-Set Wall) against 1,819 forbidden validation/test IDs.
- Verifies zero test question contamination (`exact_match_count == 0`, no unreviewed 13-gram overlaps).
- Exports frozen training corpus to `data/processed/sft_round2/train_round2.jsonl`, `train_round2.parquet`, and `manifest.json` with cryptographic SHA-256 checksums.

In [ ]:
print('=== Day 35: Prepare & Freeze Round-2 SFT Dataset ===')
if HAS_TORCH and torch.cuda.is_available():
    print('Freezing full Round-2 SFT dataset on GPU/Drive runtime...')
    !python scripts/freeze_round2_dataset.py \
        --curated_file 'data/synthetic/curated/traces_curated_sft.jsonl' \
        --targeted_file 'data/targeted/targeted_examples.jsonl' \
        --output_dir 'data/processed/sft_round2' \
        --duplicate_targeted \
        --seed 42
else:
    print('Freezing Round-2 SFT dataset in sample/mock mode (CPU / local)...')
    !python scripts/freeze_round2_dataset.py \
        --sample_mode \
        --output_dir 'data/processed/sft_round2' \
        --duplicate_targeted


## Step 33: Phase 5 Day 36 — SFT Round 2: Retrain Base Model on Combined Targeted Dataset

Retrains the base model (`deepseek-ai/deepseek-math-7b-base`) on the frozen Round-2 dataset (`data/processed/sft_round2/train_round2.jsonl`, 440 examples):
- LoRA configuration: rank $r=16, \alpha=32$, dropout $0.05$, target modules (q, k, v, o, gate, up, down).
- Learning rate: $1\times 10^{-4}$ (optimal lr from Day 28).
- Effective batch size: 16 (batch 4 $\times$ grad accum 4).
- Completion-only loss masking and canonical answer formatting.
- Automatic checkpoint resumption from `checkpoints/sft_targeted` across Colab session disconnects.
- Exports final trained adapter to `outputs/sft_targeted/adapter_model`.
- Generates audit reports (`reports/day36_round2_sft_report.md` & `.json`) and training loss curves (`reports/day36_round2_sft_curves.png`).

In [ ]:
print('=== Day 36: SFT Round 2 Training on Combined Dataset ===')
if HAS_TORCH and torch.cuda.is_available():
    print('Launching Round-2 7B SFT training on GPU (Colab G4/A100)...')
    !python scripts/train_round2_sft.py \
        --model_name_or_path 'deepseek-ai/deepseek-math-7b-base' \
        --train_file 'data/processed/sft_round2/train_round2.jsonl' \
        --val_file 'data/splits/val.parquet' \
        --output_dir 'checkpoints/sft_targeted' \
        --adapter_dir 'outputs/sft_targeted/adapter_model' \
        --learning_rate 1e-4 \
        --eval_steps 50 \
        --eval_limit 100 \
        --profile colab
else:
    print('Running Day 36 Round-2 SFT training in sample/mock mode (CPU / local)...')
    !python scripts/train_round2_sft.py \
        --sample_mode \
        --max_steps 5 \
        --eval_steps 2 \
        --eval_limit 5 \
        --output_dir 'checkpoints/sft_targeted' \
        --adapter_dir 'outputs/sft_targeted/adapter_model'


## Step 34: Phase 5 Day 37 — Compare SFT Rounds (Gating Evaluation)

Compares Round-1 adapter (`outputs/sft/adapter_model`) and Round-2 adapter (`outputs/sft_targeted/adapter_model`) on the gating split (`data/processed/val/val_gating.parquet`, 250 problems) using greedy decoding:
- Evaluates greedy accuracy, format compliance (`#### <number>`), truncation rate, and average tokens.
- Performs problem-level breakdown (mutual pass, mutual fail, Round-2 error recoveries, Round-1 regressions).
- Applies Gating Decision Rule: a 1–2 point difference ($\le 5$ problems / $\le 2.0\%$) is treated as a tie favoring Round 2.
- Stages winning adapter in `outputs/sft_targeted/adapter_model` for Phase 6 (GRPO) handoff.
- Strict Test-Set Wall isolation: test split remains completely untouched.
- Generates comparison reports (`reports/round_comparison.md` & `reports/round_comparison.json`) and plot.

In [ ]:
print('=== Day 37: Compare SFT Rounds on Gating Split ===')
if HAS_TORCH and torch.cuda.is_available():
    print('Launching Round Comparison on GPU (Colab G4/A100)...')
    !python scripts/compare_sft_rounds.py \
        --base_model 'deepseek-ai/deepseek-math-7b-base' \
        --round1_adapter 'outputs/sft/adapter_model' \
        --round2_adapter 'outputs/sft_targeted/adapter_model' \
        --val_split 'val_gating' \
        --backend auto \
        --tie_margin_pct 2.0 \
        --tie_margin_count 5 \
        --profile colab
else:
    print('Running Day 37 Round Comparison in sample/mock mode (CPU / local)...')
    !python scripts/compare_sft_rounds.py \
        --sample_mode \
        --val_split 'val_gating' \
        --limit 20 \
        --tie_margin_pct 2.0 \
        --tie_margin_count 5


## Step 35: Phase 5 Day 38 — SFT Round 2 Category Shift Analysis

Reruns error classification on Round 2's validation analysis-half errors (`data/processed/val/val_analysis.parquet`, 250 problems):
- Evaluates Round-2 LoRA adapter (`outputs/sft_targeted/adapter_model`) on the analysis split with greedy decoding.
- Classifies errors into the established taxonomy (`parse_error`, `truncation`, `arithmetic_slip`, `misread_question`, `wrong_setup`, `incomplete_reasoning`, `other`).
- Performs before-and-after frequency comparison against Round 1 baseline (`reports/error_analysis_sft.json`).
- Tracks problem-level error transition dynamics (recoveries vs incidental regressions).
- Appends Section 5 to `reports/error_analysis_sft.md` and generates `reports/category_shift_analysis.json` & `reports/day38_category_shift_report.md`.
- Strict Test-Set Wall enforcement: test split remains completely untouched.

In [ ]:
print('=== Day 38: SFT Round 2 Category Shift Analysis ===')
if HAS_TORCH and torch.cuda.is_available():
    print('Launching Category Shift Analysis on GPU (Colab G4/A100)...')
    !python scripts/run_day38_shift_analysis.py \
        --base_model 'deepseek-ai/deepseek-math-7b-base' \
        --round2_adapter 'outputs/sft_targeted/adapter_model' \
        --val_analysis_file 'data/processed/val/val_analysis.parquet' \
        --backend auto \
        --profile colab
else:
    print('Running Day 38 Category Shift Analysis in sample/mock mode (CPU / local)...')
    !python scripts/run_day38_shift_analysis.py \
        --sample_mode \
        --limit 20


## Step 36: Phase 5 Day 39 — Phase 5 Gate Verification & Release v0.6 Wrap-Up

Executes the Phase 5 Gate verification protocol to certify all exit criteria for release `v0.6`:
- **Criterion 1 (Advancing Policy & Parity):** Round 2 performance on gating split (`val_gating.parquet`, 250 problems) satisfies Gating Decision Rule (>= Round 1 - 2.0%).
- **Criterion 2 (Format Compliance):** Canonical `#### <number>` compliance >= 98.0% (Design Rule 1).
- **Criterion 3 (Truncation Rate):** Truncation rate <= 5.0% under sequence budget (512 tokens).
- **Criterion 4 (Test-Set Wall):** Strict zero test leakage; test split (`data/processed/test/test.parquet`, 1,319 problems) remains completely isolated and unqueried (Design Rule 2).
- **Criterion 5 (Category Shift):** Targeted error mitigation confirmed with net positive/neutral error suppression and regression containment.
- **Criterion 6 (Advancing Adapter):** Staged advancing policy in `outputs/sft_targeted/adapter_model` verified for Phase 6 (GRPO) hand-off.
- **Criterion 7 (Round-2 Dataset Freeze):** Cryptographic manifest and SHA-256 integrity verified for `data/processed/sft_round2/`.
- Generates `reports/phase5_wrapup_report.md` and `reports/phase5_summary.json`.


In [ ]:
print('=== Day 39: Phase 5 Gate Verification & Release v0.6 Wrap-Up ===')
!python scripts/verify_phase5_gate.py \
    --reports_dir 'reports' \
    --adapter_dir 'outputs/sft_targeted/adapter_model' \
    --dataset_dir 'data/processed/sft_round2' \
    --splits_dir 'data/splits' \
    --output_report 'reports/phase5_wrapup_report.md' \
    --output_json 'reports/phase5_summary.json'


## Step 37: Phase 6 Day 40 — GRPO Reward Functions & Advantage Computation

Initializes and validates the reinforcement learning reward architecture for GRPO (Group Relative Policy Optimization):
- **Accuracy Reward ($R_{\text{acc}}$):** Binary mathematical equivalence check against gold answer using tiered extraction.
- **Format Reward ($R_{\text{fmt}}$):** Strict Canonical Answer Contract enforcement (`\n#### <number>`).
- **Reasoning Structure Reward ($R_{\text{struct}}$):** Derivation step verification preventing guess-hacking shortcuts.
- **Length / Repetition Penalty:** Penalizes degenerate looping and excessive verbosity.
- **Group-Relative Advantages:** Verifies within-group normalization $A_i = (r_i - \mu_g) / (\sigma_g + \epsilon)$ with zero-variance safety ($A_i = 0$ when $\sigma_g = 0$).
- Generates `reports/day40_reward_design.json` and `reports/day40_reward_design.md`.


In [ ]:
print('=== Day 40: GRPO Reward Functions & Group Advantage Verification ===')
!python scripts/verify_reward_functions.py \
    --config 'configs/reward.yaml' \
    --output_dir 'reports'


## Step 38: Phase 6 Day 41 — GRPO Dataset Preparation & Difficulty Filtering

Profiles and filters training problems to maximize policy advantage variance for Group Relative Policy Optimization (GRPO):
- **Difficulty Stratification:** Rejects trivial all-pass problems (pass_rate > 0.80 or <= 2 reasoning steps) and impossible all-fail problems (pass_rate < 0.20 or > 6 reasoning steps) that cause zero-variance gradient collapse.
- **Target Difficulty Zone:** Curates intermediate problems (0.20 <= pass_rate <= 0.80 or 3-6 steps) to ensure heterogeneous group candidate outcomes under $G=4$ rollouts.
- **Strict Test-Set Wall:** Asserts 100% zero overlap with the 1,319 GSM8K test problems (`exact_matches_test: 0, forbidden_id_leaks: 0`).
- **Corpus Freezing:** Emits `data/processed/grpo/grpo_train.parquet`, `grpo_val.parquet`, `manifest.json`, and `reports/day41_grpo_dataset_report.md`.


In [ ]:
print('=== Day 41: GRPO Dataset Preparation & Difficulty Filtering ===')
!python scripts/prepare_grpo_dataset.py \
    --config 'configs/grpo_dataset.yaml' \
    --output-dir 'data/processed/grpo' \
    --report-dir 'reports'


## Step 39: Phase 6 Day 42 — Reference Model Management & KL Divergence Tracking

Initializes and validates the frozen reference policy $\pi_{\text{ref}}$ and Kullback-Leibler (KL) divergence monitoring:
- **Reference Policy Anchor:** Anchored to the advancing Phase 5 Round 2 SFT adapter (`outputs/sft_targeted/adapter_model`).
- **Schulman Estimator (k3):** Verifies the non-negative unbiased estimator $D_{\text{KL}}^{\text{k3}} = e^{-\delta_t} + \delta_t - 1 \ge 0$, guarding against negative token penalties.
- **Completion-Only Masking:** Ensures prompt and padding tokens have zero contribution to the KL penalty.
- **KL Controller & Beta Tracking:** Enforces fixed $\beta = 0.04$ and tests adaptive proportional controller with drift ceiling ($D_{\text{KL}} \le 0.50$).
- **Single-Model PEFT Memory Architecture:** Evaluates reference logits without duplicating base model weights in VRAM, fitting within Colab 15-16GB VRAM bounds.
- Emits `reports/day42_reference_model_report.md` and `reports/day42_reference_model_report.json`.


In [ ]:
print('=== Day 42: Reference Model & KL Divergence Verification ===')
!python scripts/verify_reference_model.py \
    --config 'configs/kl_divergence.yaml' \
    --output_dir 'reports'


## Step 40: Phase 6 Day 43 — GRPO Rollout Generation & Policy Update Engine

Validates the end-to-end Group Relative Policy Optimization (GRPO) rollout generation and policy update mechanics:
- **Group Rollout Generation:** Generates $G = 4$ candidate completions per mathematical prompt with sampling temperature $T = 0.7$ and nucleus top-$p = 0.95$.
- **Composite Reward & Group Normalization:** Evaluates multi-component reward $R(y) = 0.70 R_{\text{acc}} + 0.15 R_{\text{format}} + 0.10 R_{\text{struct}} + 0.05 R_{\text{len}}$ and normalizes group advantages:
  $$A_i = \frac{R_i - \bar{R}}{\sigma_R + \epsilon}$$
- **Length-Normalized Clipped Policy Gradient Loss:** Implements DeepSeekMath per-completion length normalization:
  $$\mathcal{L}_{\text{PG}}(\theta) = -\frac{1}{G}\sum_{i=1}^G \frac{1}{|o_i|}\sum_{t=1}^{|o_i|} \min\left( r_{i,t}(\theta)A_i, \text{clip}(r_{i,t}(\theta), 1-\epsilon, 1+\epsilon)A_i \right)$$
- **Reference KL Penalty Integration:** Incorporates Schulman $k_3$ estimator ($\beta = 0.04$):
  $$\mathcal{L}_{\text{GRPO}}(\theta) = \mathcal{L}_{\text{PG}}(\theta) + \beta \cdot D_{\text{KL}}^{\text{k3}}$$
- **Zero-Advantage Invariance:** Verifies mathematical identity that groups with uniform reward yield zero gradient update ($A_i = 0 \implies \mathcal{L}_{\text{PG}} = 0$).
- Emits `reports/day43_grpo_engine_report.md` and `reports/day43_grpo_engine_report.json`.

In [ ]:
print('=== Day 43: GRPO Rollout Generation & Policy Update Engine ===')
!python scripts/verify_grpo_engine.py \
    --config 'configs/grpo_args.json' \
    --report_dir 'reports'